# LSTM Code Summary

# Install Dependencies



In [ ]:
!pip install numpy==1.26.4

: 

In [ ]:

%pip install torch==2.2.0 transformers==4.46.0 tokenizers==0.20.3 sentencepiece==0.1.99 tqdm==4.65

ERROR: Could not find a version that satisfies the requirement torch==2.2.0 (from versions: 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0)
ERROR: No matching distribution found for torch==2.2.0
Note: you may need to restart the kernel to use updated packages.


# Tokenize

In [ ]:
!python get_codet5_embeddings.py --input train_code.txt \
--output train_code.pt
!python get_codet5_embeddings.py --input train_summary.txt \
--output train_summary.pt --max_length 128

!python get_codet5_embeddings.py --input val_code.txt \
--output val_code.pt
!python get_codet5_embeddings.py --input val_summary.txt \
--output val_summary.pt --max_length 128


: 

# Load Dataset into Notebook

In [82]:
#!pip install torch

In [83]:
import torch
import torch.nn as nn

data = torch.load("train_code.pt")

# plug directly into your LSTM
embedding_layer = nn.Embedding.from_pretrained(
data['embedding_matrix'], freeze=False)

# feed into the LSTM
token_ids = data['token_ids']

# Useful constants
pad_id = data['pad_token_id']
eos_id = data['eos_token_id']
vocab_size = data['vocab_size']
embedding_dim = data['embedding_dim']

# Configuration

In [97]:
BATCH_SIZE = 64
MAX_LEN = 120
HIDDEN_DIM = 512
NUM_EPOCHS = 100
EVAL_STEPS = 2000 # increased to shorten training time
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(DEVICE)

cuda


# Dataset and DataLoader

In [98]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch

class SummaryDataset(Dataset):
    def __init__(self, code_ids, summary_ids):
        self.code_ids = code_ids
        self.summary_ids = summary_ids

    def __len__(self):
        return len(self.code_ids)

    def __getitem__(self, idx):
        return (torch.tensor(self.code_ids[idx], dtype=torch.long),
            torch.tensor(self.summary_ids[idx], dtype=torch.long))

def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=pad_id)
    tgt_batch = pad_sequence(tgt_batch, batch_first=True, padding_value=pad_id)

    return src_batch, tgt_batch

# Load train data
train_code = torch.load("train_code.pt")['token_ids']
train_summary = torch.load("train_summary.pt")['token_ids']

train_dataset = SummaryDataset(train_code, train_summary)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

# Load validation data
val_code = torch.load("val_code.pt")['token_ids']
val_summary = torch.load("val_summary.pt")['token_ids']

# Create validation dataset
val_dataset = SummaryDataset(val_code, val_summary)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)



# LSTM Model

In [99]:
class Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, pad_id, sos_id, eos_id, pretrained_embeddings):
        super().__init__()
        self.embed = nn.Embedding.from_pretrained(
        pretrained_embeddings, freeze=False, padding_idx=pad_id)
        self.projection = None

        self.dropout = nn.Dropout(0.2)
        self.encoder = nn.LSTM(HIDDEN_DIM, HIDDEN_DIM, 2, batch_first=True, dropout=0.2)
        self.decoder = nn.LSTM(HIDDEN_DIM, HIDDEN_DIM, 2, batch_first=True, dropout=0.2)
        self.out = nn.Linear(HIDDEN_DIM, vocab_size)
        self.projection = nn.Linear(embedding_dim, HIDDEN_DIM)

        self.sos_id = sos_id
        self.eos_id = eos_id

    def _embed(self, x):
        e = self.embed(x)
        e = self.projection(e)
        return self.dropout(e)

    def forward(self, src, tgt):
        _, hidden = self.encoder(self._embed(src))
        output, _ = self.decoder(self._embed(tgt), hidden)
        return self.out(output)

    def generate(self, src):
        _, hidden = self.encoder(self._embed(src))
        outputs = []

        input_tok = torch.tensor([[self.sos_id]]).to(src.device)

        for _ in range(MAX_LEN):
            out, hidden = self.decoder(self._embed(input_tok), hidden)
            pred = self.out(out).argmax(-1)

            if pred.item() == self.eos_id:
                break

            outputs.append(pred.item())
            input_tok = pred

        return outputs

# Evaluation

In [100]:
!pip install sacrebleu


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [104]:
from sacrebleu.metrics import BLEU

def compute_bleu(model, val_loader, tokenizer, max_samples=500):
    model.eval()
    predictions = []
    references = []
    seen = 0

    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)

            batch_preds = []
            for i in range(src.size(0)):
                pred_ids = model.generate(src[i:i+1])
                batch_preds.append(pred_ids)

            for i in range(src.size(0)):
                if seen >= max_samples:
                    break
                pred_text = tokenizer.decode(
                    batch_preds[i],
                    skip_special_tokens=True)
                gold_text = tokenizer.decode(
                    tgt[i].tolist(),
                    skip_special_tokens=True)
                predictions.append(pred_text)
                references.append(gold_text)
                seen += 1
            if seen >= max_samples:
                break
    bleu_metric = BLEU(max_ngram_order=1)
    bleu = bleu_metric.corpus_score(predictions, [references])            
    '''bleu = sacrebleu.corpus_bleu(
        predictions,
        [references], n_gram=1 # only want bleu1 for early stopping
    )'''

    return bleu.score


# Train Model

In [106]:
import os
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm
import sacrebleu
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m")


best_bleu = 0
patience = 0

sos_id = 0 if data['pad_token_id'] != 0 else 1

model = Model(
    vocab_size=data['vocab_size'],
    embedding_dim=data['embedding_dim'],
    pad_id=data['pad_token_id'],
    eos_id=data['eos_token_id'],
    sos_id=sos_id,
    pretrained_embeddings=data['embedding_matrix']
).to(DEVICE)

save_dir = "checkpoints"
os.makedirs(save_dir, exist_ok=True)
save_name = f"lstm_codet5_small"

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4) # lower learning rate


steps_per_epoch = len(train_loader)
total_steps = NUM_EPOCHS * steps_per_epoch

step = 0
running_loss = 0
LOG_STEPS = 50

progress = tqdm(total=total_steps, desc="Training")

for epoch in range(NUM_EPOCHS):
    model.train()

    for src, tgt in train_loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)

        optimizer.zero_grad()

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        logits = model(src, tgt_input)

        loss = criterion(
            logits.reshape(-1, vocab_size),
            tgt_output.reshape(-1)
        )

        loss.backward()
        clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        step += 1
        running_loss += loss.item()

        if step % LOG_STEPS == 0:
            avg_loss = running_loss / LOG_STEPS
            progress.set_postfix({"loss": f"{avg_loss:.3f}", "epoch": epoch + 1})
            running_loss = 0

        progress.update(1)

        if step % EVAL_STEPS == 0:
            bleu = compute_bleu(model, val_loader, tokenizer, max_samples=100)
            tqdm.write(f"Step {step}/{total_steps} | BLEU-1 = {bleu:.4f}")

            if bleu > best_bleu:
                best_bleu = bleu
                patience = 0
                torch.save(model.state_dict(), f"{save_dir}/{save_name}.pt")
                tqdm.write("Saved best model")
            else:
                patience += 1
                tqdm.write(f"No improvement | Patience {patience}/10")
                if patience >= 10:
                    tqdm.write(f"\nEarly stopping at step {step}! Best BLEU-1: {best_bleu:.4f}")
                    progress.close()
                    break

            model.train()
    else:
        continue
    break

progress.close()
print(f"\nBest BLEU-1: {best_bleu:.4f}")

Training:   3%|▎         | 2000/78200 [03:28<1:52:00, 11.34it/s, loss=4.309, epoch=3]

Step 2000/78200 | BLEU-1 = 2.9899


Training:   3%|▎         | 2001/78200 [03:29<7:17:52,  2.90it/s, loss=4.309, epoch=3]

Saved best model


Training:   5%|▌         | 4000/78200 [06:54<1:56:55, 10.58it/s, loss=3.695, epoch=6]

...

Step 50000/78200 | BLEU-1 = 18.8189
No improvement | Patience 8/10


Training:  66%|██████▋   | 52000/78200 [1:29:46<38:29, 11.34it/s, loss=0.792, epoch=67]  

Step 52000/78200 | BLEU-1 = 19.3728
No improvement | Patience 9/10


Training:  69%|██████▉   | 54000/78200 [1:33:14<41:47,  9.65it/s, loss=0.742, epoch=70]  

Step 54000/78200 | BLEU-1 = 18.9506
No improvement | Patience 10/10

Early stopping at step 54000! Best BLEU-1: 20.9352

Best BLEU-1: 20.9352


# BLEU 1,2,3,4

In [110]:
!pip install pandas

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.7 MB 10.7 MB/s eta 0:00:01
   ----------------- ---------------------- 4.2/9.7 MB 10.9 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.7 MB 10.0 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.7 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 10.1 MB/s eta 0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [121]:
import pandas as pd
import torch
import sacrebleu
from tqdm import tqdm
from sacrebleu.metrics import BLEU

# load test file
df = pd.read_csv("test_dataset_tokenized.csv")

# Convert references to CodeT5 subword tokens
references_tokens = []
for ref in df["summary_tokens"]:
    ref_ids = tokenizer.encode(ref, add_special_tokens=False)
    ref_tokens = [tokenizer._convert_id_to_token(id) for id in ref_ids]
    references_tokens.append([" ".join(ref_tokens)])

# load best model
model = Model(
    vocab_size=data['vocab_size'],
    embedding_dim=data['embedding_dim'],
    pad_id=data['pad_token_id'],
    eos_id=data['eos_token_id'],
    sos_id=sos_id,
    pretrained_embeddings=data['embedding_matrix']
).to(DEVICE)

model.load_state_dict(torch.load("checkpoints/lstm_codet5_small.pt", map_location=DEVICE))
model.eval()

predictions = []

predictions_tokens = []
for ids in tqdm(df["code_ids"]):
    src = torch.tensor(eval(ids)).unsqueeze(0).to(DEVICE)
    pred_ids = model.generate(src)

    # convert predicted ids to subword tokens
    pred_tokens = [tokenizer._convert_id_to_token(i) for i in pred_ids]
    predictions_tokens.append(" ".join(pred_tokens))

# Convert references to subword tokens (same as predictions)
references_tokens = []
for ref in df["summary_tokens"]:
    ref_ids = tokenizer.encode(ref, add_special_tokens=False)
    ref_tokens = [tokenizer._convert_id_to_token(i) for i in ref_ids]
    references_tokens.append([" ".join(ref_tokens)])  # sacrebleu expects list of lists

# Compute BLEU
bleu1 = BLEU(max_ngram_order=1).corpus_score(predictions_tokens, references_tokens)
bleu2 = BLEU(max_ngram_order=2).corpus_score(predictions_tokens, references_tokens)
bleu3 = BLEU(max_ngram_order=3).corpus_score(predictions_tokens, references_tokens)
bleu4 = BLEU(max_ngram_order=4).corpus_score(predictions_tokens, references_tokens)

print("BLEU-1:", bleu1.score)
print("BLEU-2:", bleu2.score)
print("BLEU-3:", bleu3.score)
print("BLEU-4:", bleu4.score)

100%|██████████| 99/99 [00:01<00:00, 93.62it/s] 

BLEU-1: 6.112226356887866
BLEU-2: 3.1809034470435544
BLEU-3: 2.0905169555026206
BLEU-4: 1.4594726822668644


# METEOR

In [130]:
!pip install nltk

  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 11.5 MB/s eta 0:00:00
Using cached click-8.3.1-py3-none-any.whl (108 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import nltk
from nltk.translate.meteor_score import single_meteor_score
import torch
from transformers import AutoTokenizer

# Download NLTK data
nltk.download('wordnet')
nltk.download('omw-1.4')

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset
df = pd.read_csv("test_dataset_tokenized.csv")

sos_id = 0 if data['pad_token_id'] != 0 else 1

# load best model
model = Model(
    vocab_size=data['vocab_size'],
    embedding_dim=data['embedding_dim'],
    pad_id=data['pad_token_id'],
    eos_id=data['eos_token_id'],
    sos_id=sos_id,
    pretrained_embeddings=data['embedding_matrix']
).to(DEVICE)

checkpoint_path = "checkpoints/lstm_codet5_small.pt"
state_dict = torch.load(checkpoint_path, map_location=DEVICE)

model.load_state_dict(state_dict)
model.eval()


# Load tokenizer from Hugging Face (same as SIDE)
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5p-220m", add_prefix_space=True)
token_ids = data["token_ids"]

PAD_ID = data["pad_token_id"]
EOS_ID = data["eos_token_id"]

# METEOR evaluation
meteor_scores = []

with torch.no_grad():
    for ids, ref_text in zip(df["code_ids"], df["summary"]):
        token_list = eval(ids)[:MAX_LEN]
        src = torch.tensor(token_list).unsqueeze(0).to(DEVICE)

        # Generate from LSTM
        pred_ids = model.generate(src)

        # Decode using tokenizer
        pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)
        if not pred_text:
            pred_text = "<empty>"

        # Tokenize for METEOR
        ref_tokens = ref_text.split()
        pred_tokens = pred_text.split()

        # Compute METEOR
        if pred_tokens == ["<empty>"]:
            meteor_scores.append(0.0)
        else:
            meteor_scores.append(single_meteor_score(ref_tokens, pred_tokens))

# Average METEOR
avg_meteor = sum(meteor_scores) / len(meteor_scores)
print("Average METEOR:", avg_meteor)

: 

# BERTScore

In [2]:
!pip install bert-score

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached filelock-3.25.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached setuptools-82.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Us

In [ ]:
import pandas as pd
import torch
import ast
from tqdm import tqdm
from bert_score import score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("test_dataset_tokenized.csv")

    
sos_id = 0 if data['pad_token_id'] != 0 else 1

model = Model(
    vocab_size=data["vocab_size"],
    embedding_dim=data["embedding_dim"],
    pad_id=data["pad_token_id"],
    eos_id=data["eos_token_id"],
    sos_id=sos_id,
    pretrained_embeddings=data["embedding_matrix"]
).to(DEVICE)

checkpoint_path = "checkpoints/lstm_codet5_small.pt"
state_dict = torch.load(checkpoint_path, map_location=DEVICE)

model.load_state_dict(state_dict)
model.eval()

token_ids = data["token_ids"]

PAD_ID = data["pad_token_id"]
EOS_ID = data["eos_token_id"]

predictions = []
references = []

with torch.no_grad():

    for ids, reference in tqdm(
        zip(df["code_ids"], df["summary"]),
        total=len(df),
        desc="Generating predictions"
    ):

        src_ids = ast.literal_eval(ids)

        src_tensor = torch.tensor(src_ids).unsqueeze(0).to(DEVICE)

        pred_ids = model.generate(src_tensor)
        pred_text = tokenizer.decode(pred_ids, skip_special_tokens=True)

        predictions.append(pred_text)
        references.append(reference)

P, R, F1 = score(
    predictions,
    references,
    lang="en",
    model_type="roberta-large"
)

print("\nBERTScore")
print(f"Precision : {P.mean().item():.4f}")
print(f"Recall    : {R.mean().item():.4f}")
print(f"F1 Score  : {F1.mean().item():.4f}")

/Users/abigailschwall/.pyenv/versions/3.12.6/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'Model' is not defined

# SIDE

In [125]:
!pip install sentence-transformers

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------- ----------------------------- 2.1/8.0 MB 10.7 MB/s eta 0:00:01
   --------------------------------- ------ 6.8/8.0 MB 16.8 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 17.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.5 MB ? eta -:--:--
   ------ --------------------------------- 5.8/36.5 MB 27.1 MB/s eta 0:00:02
   ------------ --------------------------- 11.8/36.5 MB 28.4 MB/s eta 0:00:01
   ------------------- -------------------- 17.6/36.5 MB 27.7 MB/s eta 0:00:01
   ------------------------- -------------- 23.3/36.5 MB 28.4 MB/s eta 0:00:01
   -------------------------------- ------- 29.6/36.5 MB 28.9 MB/s eta 0:00:01
   ---------------------------------------  35.9/36.5 MB 28.9 MB/s eta 0:00:01
   ---------------------------------------- 36.5/36.5 MB 27.3 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from sentence_transformers import util
from transformers import AutoTokenizer, AutoModel, T5EncoderModel
import torch
import torch.nn.functional as F

# Mean Pooling - Take attention mask into account
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

# Use encoder-only T5 model
checkPointFolder = "Salesforce/codet5p-220m"
tokenizer = AutoTokenizer.from_pretrained(checkPointFolder)
side_model = T5EncoderModel.from_pretrained(checkPointFolder).to(DEVICE)
side_model.eval()

sos_id = 0 if data['pad_token_id'] != 0 else 1

# load lstm model
lstm_model = Model(
    vocab_size=data['vocab_size'],
    embedding_dim=data['embedding_dim'],
    pad_id=data['pad_token_id'],
    eos_id=data['eos_token_id'],
    sos_id=sos_id,
    pretrained_embeddings=data['embedding_matrix']
).to(DEVICE)

lstm_model.load_state_dict(torch.load("checkpoints/lstm_codet5_small.pt", map_location=DEVICE))
lstm_model.eval()

# Example code snippet and summary
method = """
  public Object pop() throws EmptyStackException {
        try {
            Object aObject = this.stackElements.get(stackElements.size() - 1);
            stackElements.remove(stackElements.size() - 1);
            return aObject;
        } catch (Exception e) {
            throw new EmptyStackException(e);
        }
    }
"""
ref_summary = "pop the top of the stack"

# generate summary with lstm
src_ids = tokenizer.encode(method)
src_tensor = torch.tensor(src_ids).unsqueeze(0).to(DEVICE)

pred_ids = lstm_model.generate(src_tensor)
pred_summary = tokenizer.decode(pred_ids)


pair = [ref_summary, pred_summary]

# Tokenize
encoded_input = tokenizer(pair, padding=True, truncation=True, return_tensors='pt').to(DEVICE)

# Compute token embeddings
with torch.no_grad():
    model_output = side_model(**encoded_input)

# Mean pooling
sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])

# Normalize embeddings
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

sim = util.pytorch_cos_sim(sentence_embeddings[0], sentence_embeddings[1]).item()

print("SIDE score: ", sim)

0.414553165435791
